# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their @id
record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
elif hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
else:
    # Try possible fallback in Croissant 1.x schema
    record_sets = getattr(metadata, 'record_sets', [])

print("Record Sets (@id and name):")
record_set_ids = []
for rs in record_sets:
    print(f"  @id: {rs.id}, name: {getattr(rs, 'name', 'N/A')}")
    record_set_ids.append(rs.id)

if not record_sets:
    print('No record sets found in the metadata. Attempting to display distribution files.')

# Some datasets store main records in distributions
if hasattr(metadata, 'distribution'):
    print("Distributions (potential data files):")
    for dist in metadata.distribution:
        print(f"  @id: {dist.id}, url: {getattr(dist, 'contentUrl', 'N/A')}, encodingFormat: {getattr(dist, 'encodingFormat', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
dataframes = {}
import warnings

if record_set_ids:
    for record_set_id in record_set_ids:
        print(f"Loading records from record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded dataframe for {record_set_id} with {df.shape[0]} records and {df.shape[1]} columns.")
        else:
            print(f"No records found for record set {record_set_id}.")
    # Show columns for the first record set if available
    first_rs = record_set_ids[0]
    if first_rs in dataframes:
        print(f"Columns in DataFrame ({first_rs}):")
        print(dataframes[first_rs].columns.tolist())
        display(dataframes[first_rs].head())
else:
    warnings.warn("No record_set entries available to load as dataframes. Data may be referenced in distribution directly.")
    # List available distribution files for manual inspection
    if hasattr(metadata, 'distribution'):
        for dist in metadata.distribution:
            print(f"Distribution @id: {dist.id}, url: {getattr(dist, 'contentUrl', None)}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# As we do not know actual record set/field ids, try with the only dataframe loaded.
import numpy as np
EDA_RS = None
if dataframes:
    EDA_RS = list(dataframes.keys())[0]
    df = dataframes[EDA_RS]
    print(f"Performing EDA on record set: {EDA_RS}")
    
    # List possible numeric fields
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields: {numeric_candidates}")
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using {numeric_field} for analysis.")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field
        group_candidates = df.select_dtypes(include=[object]).columns.tolist()
        group_field = None
        for col in group_candidates:
            # Heuristically select the first with <10 unique values that isn't an ID
            if col.lower().endswith('id'):
                continue
            if df[col].nunique() < 10:
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (showing average {numeric_field}):")
            display(grouped_df)
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("No numeric fields found for analysis.")
else:
    print("No dataframe loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and EDA_RS and numeric_candidates:
    # Plot histogram of selected numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If a group field was found, show boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric data available for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

_In this notebook, we demonstrated how to access and load metadata and records from a Croissant-described dataset using `mlcroissant`. We explored available record sets, fields, extracted data into pandas DataFrames, applied data filtering and normalization, and visualized field distributions. This approach enables fast, reproducible data exploration directly from FAIR datasets described in Croissant._